# ⚡ NanoRecall: 100% Private Desktop Memory & Screen Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminsk/nanorecall/blob/main/notebooks/nanorecall_quickstart.ipynb)
[![GitHub Stars](https://img.shields.io/github/stars/eminsk/nanorecall?style=flat-square&color=00f2fe)](https://github.com/eminsk/nanorecall)
[![Powered by NanoVector](https://img.shields.io/badge/powered%20by-NanoVector%20AVX2-4facfe?style=flat-square)](https://github.com/eminsk/nanovector)
[![Ubuntu / Debian PPA](https://img.shields.io/badge/Ubuntu%20%2F%20Debian-APT%20PPA-E95420?logo=ubuntu&logoColor=white)](https://eminsk.github.io/ppa/)
[![Conda-Forge](https://img.shields.io/conda/vn/conda-forge/nanorecall.svg)](https://anaconda.org/conda-forge/nanorecall)
[![License](https://img.shields.io/badge/license-MIT-green.svg?style=flat-square)](https://github.com/eminsk/nanorecall/blob/main/LICENSE)

**NanoRecall** is a high-performance, open-source, 100% private alternative to **Microsoft Windows Recall**.
Engineered with pure C99/SIMD, native Win32 capture, and zero cloud telemetry. Search anything you saw on your screen by natural language in milliseconds without an NPU or cloud subscription.

### 🌟 Key Highlights Demonstrated in this Notebook:
- **0% Cloud Telemetry:** Everything runs locally on CPU with zero internet egress.
- **No NPU Required:** Powered by [NanoVector](https://github.com/eminsk/nanovector)'s AVX2/NEON SIMD vector search kernel.
- **Built-in Privacy Shield:** Auto-ignores password managers and redacts API keys and credit cards.
- **Smart Frame Differencing:** Skips duplicate captures when screen is static.
- **Sub-Millisecond Search:** Instant semantic recall (<0.3 ms) over your computer history.

## 1. 📦 Installation

Install `nanorecall` and `nanovector` directly in Google Colab:

> 💡 **For Ubuntu 24.04 / Debian systems (outside Colab)**: You can also install system-wide via our official PPA:
> `curl -sS https://eminsk.github.io/ppa/setup.sh | sudo bash && sudo apt install -y python3-nanorecall python3-nanovector`


In [ ]:
# 🚀 1. Удаляем старые debian-пакеты (если были установлены):
!sudo apt-get remove -y python3-nanorecall python3-nanovector >/dev/null 2>&1 || true

# 2. Установка NanoRecall и NanoVector C99 AVX2:
!pip install -q --no-cache-dir --ignore-installed nanovector nanorecall pillow numpy

# 3. Очистка кэша импортов:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith('nanovector') or mod.startswith('nanorecall'):
        del sys.modules[mod]

import nanorecall
import nanovector
print('=' * 60)
print(f'✅ NanoRecall v{nanorecall.__version__} & NanoVector v{nanovector.__version__} успешно загружены!')
print(f'⚡ NanoVector Backend: {nanovector.__backend__}')
print('=' * 60)


## 2. ⚡ Verification & Backend Inspection

Verify the installed packages and inspect the active hardware acceleration (AVX2/NEON):

In [ ]:
import nanorecall
import nanovector

print(f"✅ NanoRecall Version: v{nanorecall.__version__}")
print(f"⚡ NanoVector Engine:  v{nanovector.version()} ({nanovector.simd_backend()})")

## 3. 🛡️ Privacy Shield Demo: Auto-Ignoring Sensitive Apps & Redaction

Unlike Microsoft Recall (which saves sensitive credentials to unencrypted SQLite databases), **NanoRecall** includes an out-of-the-box **Privacy Shield**.
It immediately terminates recording when password managers or private windows are in focus, and redacts sensitive strings from OCR text:

In [ ]:
from nanorecall import PrivacyShield

shield = PrivacyShield()

# 1. Test Window Blacklist Filtering
test_windows = [
    ("1Password — Personal Vault", "Chrome_WidgetWin_1"),
    ("KeePassXC - Passwords.kdbx", "Qt5QWindowIcon"),
    ("Google Chrome - Incognito Tab", "Chrome_WidgetWin_1"),
    ("MetaMask Notification", "Chrome_WidgetWin_1"),
    ("Visual Studio Code - nanorecall/memory.py", "Code"),
    ("GitHub: Where the world builds software", "Chrome_WidgetWin_1"),
    ("PowerShell - Terminal", "ConsoleWindowClass")
]

print("🛡️ Active Window Privacy Filter Checks:")
for title, cls in test_windows:
    is_private = shield.is_window_private(title, cls)
    status = "🚫 BLOCKED (Shielded)" if is_private else "✅ ALLOWED"
    print(f"  {status:<25} -> {title}")

# 2. Test Credential & Token Redaction
sample_text = "Logged in with key sk-9876543210abcdef1234567890abcdef and card 4111 2222 3333 4444 on terminal."
redacted_text = shield.redact_text(sample_text)
print("\n🔒 Text Redaction Verification:")
print(f"  Original: {sample_text}")
print(f"  Redacted: {redacted_text}")

## 4. 🖥️ Desktop Session Simulation: Generating Sample Screen Captures

To demonstrate screen recall in this headless Colab environment, let's synthesize 4 realistic desktop screenshot frames (Chrome reading GitHub PR, Terminal debugging Docker, IDE coding Python, and a recipe note) using Pillow:

In [ ]:
import os
from pathlib import Path
from PIL import Image, ImageDraw

frames_dir = Path("./sample_frames")
frames_dir.mkdir(exist_ok=True)

sample_sessions = [
    {
        "id": "frame_001",
        "app": "Google Chrome",
        "title": "sqlfluff pull request 8449 github",
        "text": "Merged upstream/main into pull request #8449. Added StarRocks test cases for CV11 syntax support.",
        "time": "2026-09-11_142315",
        "bg_color": (22, 27, 34),       # GitHub dark mode
        "text_color": (88, 166, 255)
    },
    {
        "id": "frame_002",
        "app": "Terminal",
        "title": "PowerShell - docker build",
        "text": "ERROR: Container failed with segmentation fault on port 8080 during memory allocation.",
        "time": "2026-09-11_151040",
        "bg_color": (12, 12, 12),       # Terminal black
        "text_color": (248, 81, 73)
    },
    {
        "id": "frame_003",
        "app": "VS Code",
        "title": "nanovector.c — Bare-Metal AVX2",
        "text": "__m256 ymm0 = _mm256_loadu_ps(&a[i]); _mm256_fmadd_ps(ymm0, ymm1, sum);",
        "time": "2026-09-11_160522",
        "bg_color": (30, 30, 30),       # VS Code dark
        "text_color": (78, 201, 176)
    },
    {
        "id": "frame_004",
        "app": "Notes",
        "title": "Weekend Dinner Recipe",
        "text": "Chocolate lava cake with dark cocoa powder, espresso shot, and vanilla bean cream.",
        "time": "2026-09-11_174500",
        "bg_color": (40, 35, 30),       # Warm notes
        "text_color": (255, 215, 0)
    }
]

# Draw synthetic screenshots
for s in sample_sessions:
    img = Image.new("RGB", (800, 450), color=s["bg_color"])
    draw = ImageDraw.Draw(img)
    # Header bar
    draw.rectangle([(0, 0), (800, 35)], fill=(15, 15, 20))
    draw.text((15, 10), f"[ {s['app']} ] — {s['title']}", fill=(220, 220, 220))
    # Content body
    draw.text((40, 80), f"Timestamp: {s['time']}", fill=(140, 140, 140))
    draw.text((40, 120), s["text"], fill=s["text_color"])
    # Save frame & thumbnail
    img_path = frames_dir / f"{s['time']}.png"
    img.save(img_path)
    s["img_path"] = str(img_path)

print(f"✅ Generated {len(sample_sessions)} realistic desktop session frames in '{frames_dir}'!")

## 5. 🧠 Indexing into NanoVector Semantic Episodic Memory

Now, we index these screen captures into **`RecallMemory`**, powered by `NanoVector`:
- Each frame's OCR text, application name, window title, and image path are vectorized into normalized embeddings.
- Everything is stored in an ultra-compact binary `.nvec` file.

In [ ]:
from nanorecall import RecallMemory

# Initialize local memory store (stores to ./nanorecall_db/memory.nvec)
db_dir = Path("./nanorecall_db")
memory = RecallMemory(db_dir=db_dir, embed_dim=128)

# Index the frames
for s in sample_sessions:
    memory.index_frame(
        frame_id=s["id"],
        text=s["text"],
        app_name=s["app"],
        window_title=s["title"],
        image_path=s["img_path"],
        timestamp=s["time"]
    )

memory.save()

stats = memory.get_stats()
print("⚡ NanoRecall Memory Telemetry:")
print(f"  Total Frames Indexed: {stats['total_frames']}")
print(f"  Vector Dimension:     {stats['vector_dim']}D")
print(f"  Active Backend:       {stats['backend']}")
print(f"  Database Size:        {stats['file_size_kb']:.2f} KB (Single .nvec file!)")

## 6. 🔍 Natural Language Semantic Screen Recall (< 1 ms)

Ask natural questions to recall anything you saw on your monitor hours or days ago.
Notice how **NanoVector** searches by meaning and returns the exact screen capture in less than **0.3 milliseconds**:

In [ ]:
import time
try:
    from IPython.display import display, Image as IPImage
except ImportError:
    display = None
    IPImage = None

def recall_desktop(query: str, top_k: int = 1):
    t0 = time.perf_counter()
    results = memory.search(query, top_k=top_k)
    latency_ms = (time.perf_counter() - t0) * 1000
    
    print(f"\n🔍 Query: '{query}' (Search Latency: {latency_ms:.3f} ms)")
    print("=" * 75)
    
    for rank, m in enumerate(results, 1):
        score_pct = int(max(0, min(100, m.score * 100)))
        print(f"#{rank} [{score_pct}% Match] App: {m.app_name} | Window: '{m.window_title}'")
        print(f"   🕒 Captured: {m.timestamp}")
        print(f"   📝 Text:     {m.snippet}")
        if m.image_path and os.path.exists(m.image_path) and display and IPImage:
            print(f"   🖼️  Screenshot Display:")
            display(IPImage(m.image_path, width=500))

# Test queries
recall_desktop("Where was the sqlfluff pull request for StarRocks?")
recall_desktop("docker container crashed with segfault")
recall_desktop("recipe for chocolate lava cake")

## 7. 💾 Single-File Persistence (.nvec) & Instant Reload

All your desktop memories are securely packed into a single binary file.
Let's simulate shutting down the process and reloading from disk in a fresh instance with zero overhead:

In [ ]:
# Load fresh instance pointing to existing database
restored_memory = RecallMemory(db_dir=db_dir, embed_dim=128)

print(f"✅ Successfully restored memory index!")
print(f"   Total Indexed Items: {len(restored_memory.index)}")

# Verify query on reloaded index
match = restored_memory.search("AVX2 assembly code", top_k=1)[0]
print(f"   Verification Query Top Match: [{match.app_name}] {match.window_title}")
print(f"   Matching Snippet: {match.snippet}")

## 8. 🚀 Latency & Throughput Benchmark on Colab CPU

Let's benchmark **NanoRecall's search latency and query throughput (QPS)** across thousands of stored desktop memories directly on this Google Colab virtual machine:

In [ ]:
import numpy as np

print("⚡ NanoRecall / NanoVector Search Latency Benchmark on Colab CPU:\n")
print(f"{'Stored Frames (N)':<20} | {'Avg Search Latency':<25} | {'Throughput (QPS)':<16}")
print("-" * 68)

bench_dir = Path("./nanorecall_bench")
bench_dir.mkdir(exist_ok=True)

for N in [500, 2000, 10000]:
    bench_mem = RecallMemory(db_dir=bench_dir / f"db_{N}", embed_dim=128)
    
    # Batch ingest N simulated memory frames
    rng = np.random.default_rng(42)
    matrix = rng.standard_normal((N, 128)).astype(np.float32)
    # L2 normalize
    matrix /= np.linalg.norm(matrix, axis=1, keepdims=True)
    
    for i in range(N):
        bench_mem.index.add(id=f"frame_{i}", vector=matrix[i], metadata='{"app":"bench"}')
        
    # Measure search speed over 100 queries
    queries = rng.standard_normal((100, 128)).astype(np.float32)
    queries /= np.linalg.norm(queries, axis=1, keepdims=True)
    
    t0 = time.perf_counter()
    for q in queries:
        bench_mem.index.search(q, top_k=5)
    t_search = (time.perf_counter() - t0) / 100
    qps = 1.0 / t_search
    
    lat_str = f"{t_search*1000:.3f} ms ({t_search*1e6:.0f} µs)"
    print(f"N = {N:<16,d} | {lat_str:>25} | {qps:>12,.0f} QPS")

print("\n⚡ Summary: Even with 10,000 recorded desktop frames, search is SUB-MILLISECOND on standard CPU!")

## 9. 🔗 Resources & Getting Started Locally

Ready to run **NanoRecall** on your personal PC?

- **GitHub Repository:** [github.com/eminsk/nanorecall](https://github.com/eminsk/nanorecall)
- **CLI Command:** `nanorecall capture` / `nanorecall search "..."`
- **Local Web Dashboard:** `nanorecall ui` (Opens dark-mode timeline dashboard at `http://127.0.0.1:8765`)
- **Core Vector Engine:** [github.com/eminsk/nanovector](https://github.com/eminsk/nanovector)

*Zero Cloud. Zero NPU. 100% Private.*